In [ ]:
# uncomment these and run this cell if needed
!pip install evaluate
!pip install huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.8 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification
from transformers import AutoTokenizer
from transformers import get_scheduler
from transformers import Trainer, TrainingArguments
from datasets import concatenate_datasets
import evaluate
import numpy as np

In [ ]:
#initalize variables
header1 = "Passage: "
header2 = "Question: "
header3 = "Answer: "

column_names1 = "passage"
column_names2 = "question"
column_names3 = "answer"

def set_header1(header_value):
  global header1
  header1=header_value

def set_header2(header_value):
  global header2
  header2=header_value

def set_header3(header_value):
  global header3
  header3=header_value

def set_column_names1(column_names):
  global column_names1
  column_names1=column_names

def set_column_names2(column_names):
  global column_names2
  column_names2=column_names

def set_column_names3(column_names):
  global column_names3
  column_names3=column_names

def set_master_columns(col1, col2, col3, header_val1, header_val2, header_val3):
  global column_names1, column_names2, column_names3, header1, header2, header3
  column_names1=col1
  column_names2=col2
  column_names3=col3

  header1=header_val1
  header2=header_val2
  header3=header_val3

In [ ]:
# prepping the dataset
raw_datasets_base = load_dataset("flowaicom/HaluEval")#https://huggingface.co/datasets/flowaicom/HaluEval
raw_datasets_medical = load_dataset("GM07/medhal-lf") #https://huggingface.co/datasets/GM07/medhal-lf

def label_map(example):
    if example["label"] == "PASS" or example["label"] == "true" or example["label"] == "1" or example["label"] == 1:
        example["label"] = 1
    else:
        example["label"] = 0
    return example

raw_datasets_base = raw_datasets_base.map(label_map)

raw_datasets_medical["train"] = raw_datasets_medical["train"].select(range(10000))
raw_datasets_medical= raw_datasets_medical.map(label_map)


# convert medhal-lf columns to match HaluEval
def convert_medical(example):
    return {
        "passage": example["context"],      # context -> passage
        "question": "What is a true medical fact?",
        "answer": example["statement"],     # statement -> answer
        "label": example["label"]
    }
raw_datasets_medical = raw_datasets_medical.map(convert_medical)

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

def tokenize_function(examples):
    global header1, header2, header3, column_names1, column_names2, column_names3
    combined = [
        (
            header1 + (q if q is not None else ""),
            header2 + (p if p is not None else "") + header3 + (a if a is not None else "")
        )
        for q, p, a in zip(examples[column_names1], examples[column_names2], examples[column_names3])
    ]

    return tokenizer(
        [c[0] for c in combined],
        [c[1] for c in combined],
        padding="max_length",
        truncation=True
    )
#[SEP] is a special token used in BERT to separate different parts of the input sequence, such as the question, context, and answer, allowing the model to distinguish between them.
#[CLS] start of input
#tokenizer does this automatically for us



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]

README.md: 0.00B [00:00, ?B/s]

...

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
from datasets import Value
from datasets import DatasetDict
cols = ["passage", "question", "answer", "label"]

base_clean = raw_datasets_base.remove_columns(
    [c for c in raw_datasets_base["test"].column_names if c not in cols]
)

medical_clean = raw_datasets_medical.remove_columns(
    [c for c in raw_datasets_medical["train"].column_names if c not in cols]
)

medical_clean = medical_clean.cast_column("label", Value("int64"))
base_clean = base_clean.cast_column("label", Value("int64"))

combined_train = concatenate_datasets([
    base_clean["test"],
    medical_clean["train"]
])

combined_dataset = DatasetDict({
    "train": combined_train
})



Casting the dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
set_master_columns("passage", "question", "answer", "Question: ", "Passage: ", "Answer: ")
tokenized_datasets = combined_dataset.map(tokenize_function, batched=True)


Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
split_datasets = tokenized_datasets["train"].train_test_split(test_size=0.2, seed=42)

In [ ]:
small_train_dataset = split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = split_datasets["test"].shuffle(seed=42).select(range(100))

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

#added average binary because it tells the model to Treat this as a binary classification problem and focus on the positive class (1)
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "f1": f1.compute(predictions=predictions, references=labels, average="binary")["f1"],
    }

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train and evalute the results for base and medical
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.656333,0.630000,0.560000,0.913043,0.694215
2,No log,0.466342,0.840000,0.894737,0.739130,0.809524
3,No log,0.282395,0.900000,0.860000,0.934783,0.895833
4,No log,0.259484,0.920000,0.895833,0.934783,0.914894
5,No log,0.289868,0.920000,0.865385,0.978261,0.918367
6,No log,0.476777,0.910000,0.877551,0.934783,0.905263
7,No log,0.444976,0.920000,0.851852,1.000000,0.920000
8,No log,0.403545,0.930000,0.867925,1.000000,0.929293


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.40354496240615845,
 'eval_accuracy': 0.93,
 'eval_precision': 0.8679245283018868,
 'eval_recall': 1.0,
 'eval_f1': 0.9292929292929293,
 'eval_runtime': 0.7779,
 'eval_samples_per_second': 128.552,
 'eval_steps_per_second': 16.712,
 'epoch': 8.0}

In [ ]:

ds = load_dataset("Cleanlab/FinQA-hallucination-detection") #https://huggingface.co/datasets/Cleanlab/FinQA-hallucination-detection

README.md: 0.00B [00:00, ?B/s]

finqa-hallucination-detection.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1657 [00:00<?, ? examples/s]

In [ ]:
#switch to a different data set for evalute
raw_eval = ds.rename_columns({
    "query": "question",
    "context": "passage",
    "llm_response": "answer",
    "is_correct": "label"
})

def truncate_context(example):
    example["passage"] = str(example["passage"])[:200]
    example["question"] = str(example["question"])[:100]
    return example

raw_eval = raw_eval.map(truncate_context)


from datasets import concatenate_datasets

# split by label
label_0 = raw_eval["train"].filter(lambda x: x["label"] == 0)
label_1 = raw_eval["train"].filter(lambda x: x["label"] == 1)

# get smallest class size
min_size = min(len(label_0), len(label_1))
print(len(label_0))
print(len(label_1))

# downsample both to same size
label_0 = label_0.shuffle(seed=42).select(range(min_size))
label_1 = label_1.shuffle(seed=42).select(range(min_size))

# combine and shuffle
balanced_eval = concatenate_datasets([label_0, label_1]).shuffle(seed=42)

balanced_eval=balanced_eval.map(tokenize_function, batched=True)




small_trai_ndataset = tokenized_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = balanced_eval.shuffle(seed=42).select(range(100))
from collections import Counter

print(Counter(small_eval_dataset["label"]))

Map:   0%|          | 0/1657 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1657 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1657 [00:00<?, ? examples/s]

239
1418


Map:   0%|          | 0/478 [00:00<?, ? examples/s]

Counter({0: 58, 1: 42})


In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train and evalute with a different test set
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.700024,0.420000,0.416667,0.952381,0.579710
2,No log,0.784913,0.440000,0.423913,0.928571,0.582090
3,No log,1.201791,0.440000,0.425532,0.952381,0.588235
4,No log,1.681344,0.440000,0.425532,0.952381,0.588235
5,No log,1.857860,0.430000,0.421053,0.952381,0.583942
6,No log,2.313023,0.440000,0.427083,0.976190,0.594203
7,No log,2.503351,0.430000,0.421053,0.952381,0.583942
8,No log,2.606956,0.430000,0.421053,0.952381,0.583942


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 2.6069557666778564,
 'eval_accuracy': 0.43,
 'eval_precision': 0.42105263157894735,
 'eval_recall': 0.9523809523809523,
 'eval_f1': 0.583941605839416,
 'eval_runtime': 0.778,
 'eval_samples_per_second': 128.542,
 'eval_steps_per_second': 16.71,
 'epoch': 8.0}

In [ ]:
#We train on base and test on medical so we can see the difference
tokenized_datasets_base = base_clean.map(tokenize_function, batched=True)
tokenized_datasets_medical = medical_clean.map(tokenize_function, batched=True)

small_train_dataset = tokenized_datasets_base["test"].shuffle(seed=42).select(range(400)) #this dataset did not have a "train" corpus, even so its called test, we used it for training
small_eval_dataset = tokenized_datasets_medical["train"].shuffle(seed=42).select(range(100)) #this dataset did not have a test subset but we wanted to test on the medical data
#the "test" and "train" look switched butt this was intented so we can evaluate on the medical

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.642106,0.630000,0.611111,0.964912,0.748299
2,No log,0.942321,0.570000,0.570000,1.000000,0.726115
3,No log,1.993584,0.570000,0.570000,1.000000,0.726115
4,No log,2.398794,0.570000,0.570000,1.000000,0.726115
5,No log,2.271745,0.570000,0.570000,1.000000,0.726115
6,No log,2.595180,0.570000,0.570000,1.000000,0.726115
7,No log,2.691852,0.570000,0.570000,1.000000,0.726115
8,No log,2.720534,0.570000,0.570000,1.000000,0.726115


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 2.720534324645996,
 'eval_accuracy': 0.57,
 'eval_precision': 0.57,
 'eval_recall': 1.0,
 'eval_f1': 0.7261146496815286,
 'eval_runtime': 0.7638,
 'eval_samples_per_second': 130.928,
 'eval_steps_per_second': 17.021,
 'epoch': 8.0}

In [ ]:
#next step medical train and medical evaluate
medical_split_datasets = tokenized_datasets_medical["train"].train_test_split(test_size=0.2, seed=42)

small_train_dataset = medical_split_datasets["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = medical_split_datasets["test"].shuffle(seed=42).select(range(100))

In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.433865,0.810000,0.730159,0.958333,0.828829
2,No log,0.313753,0.880000,0.846154,0.916667,0.880000
3,No log,0.383434,0.880000,0.928571,0.812500,0.866667
4,No log,0.305050,0.900000,0.851852,0.958333,0.901961
5,No log,0.389993,0.910000,0.914894,0.895833,0.905263
6,No log,0.334335,0.930000,0.918367,0.937500,0.927835
7,No log,0.298604,0.940000,0.920000,0.958333,0.938776
8,No log,0.353733,0.920000,0.916667,0.916667,0.916667


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.35373252630233765,
 'eval_accuracy': 0.92,
 'eval_precision': 0.9166666666666666,
 'eval_recall': 0.9166666666666666,
 'eval_f1': 0.9166666666666666,
 'eval_runtime': 0.7724,
 'eval_samples_per_second': 129.47,
 'eval_steps_per_second': 16.831,
 'epoch': 8.0}

In [ ]:
#no training
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

small_eval_dataset = medical_split_datasets["test"].shuffle(seed=42).select(range(100))
#train
trainer = Trainer(
    model=model, args=training_args, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)

# evaluate WITHOUT training
trainer.evaluate()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


{'eval_loss': 0.7262813448905945,
 'eval_model_preparation_time': 0.0036,
 'eval_accuracy': 0.49,
 'eval_precision': 0.48484848484848486,
 'eval_recall': 1.0,
 'eval_f1': 0.6530612244897959,
 'eval_runtime': 1.2839,
 'eval_samples_per_second': 77.89,
 'eval_steps_per_second': 10.126}

In [ ]:
#next step see what happens when parameters are changed
set_master_columns("question", "passage", "answer", "Passage: ", "Question: ", "Answer: ")

tokenized_datasets_medical_2 = medical_clean.map(tokenize_function, batched=True)
medical_split_datasets_2 = tokenized_datasets_medical_2["train"].train_test_split(test_size=0.2, seed=42)


small_train_dataset = medical_split_datasets_2["train"].shuffle(seed=42).select(range(400))
small_eval_dataset = medical_split_datasets_2["test"].shuffle(seed=42).select(range(100))

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [ ]:
print(small_train_dataset.column_names)

['label', 'passage', 'question', 'answer', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
#setting up the model

#default
model = AutoModelForSequenceClassification.from_pretrained("bert-base-cased", num_labels=2)

training_args = TrainingArguments(eval_strategy="epoch", num_train_epochs = 8, learning_rate = 2e-5)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
#train
trainer = Trainer(
    model=model, args=training_args, train_dataset=small_train_dataset, eval_dataset=small_eval_dataset, compute_metrics=compute_metrics,
)
trainer.train()
trainer.evaluate()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,No log,0.692602,0.540000,0.533333,0.333333,0.410256
2,No log,0.692593,0.520000,0.500000,0.208333,0.294118
3,No log,0.705595,0.500000,0.416667,0.104167,0.166667
4,No log,0.709874,0.470000,0.444444,0.416667,0.430108
5,No log,0.714511,0.540000,0.522727,0.479167,0.500000
6,No log,0.738156,0.480000,0.472222,0.708333,0.566667
7,No log,0.738253,0.540000,0.516667,0.645833,0.574074
8,No log,0.762989,0.510000,0.492958,0.729167,0.588235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.762988805770874,
 'eval_accuracy': 0.51,
 'eval_precision': 0.49295774647887325,
 'eval_recall': 0.7291666666666666,
 'eval_f1': 0.5882352941176471,
 'eval_runtime': 0.7664,
 'eval_samples_per_second': 130.474,
 'eval_steps_per_second': 16.962,
 'epoch': 8.0}